In [41]:
import numpy as np
import pandas as pd
import json

from typing import Annotated
from pydantic import BaseModel
from pydantic.json import pydantic_encoder

from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers.json import JsonOutputParser
from langchain_ollama  import ChatOllama, OllamaEmbeddings

gpt = ChatOllama(
    model    = "gpt-oss:20b", 
    base_url = "http://localhost:11434",
)
qwen= OllamaEmbeddings(
    model    = "qwen3:8b",
    base_url = "http://localhost:11434",
)

class QA(BaseModel):
    """Question/Response to the customer service."""

    question: Annotated[str, "The question asked by the user"]
    response: Annotated[str, "The response from customer support"]

class ModelResponse(BaseModel):
    """Typesafe list of question/responses"""
    
    data: Annotated[list[QA], "List of question/responses"]

template = ChatPromptTemplate.from_template(
    """
    Below are three examples of emails that my customer support
    has received. As I want to train an AI to process and triage
    such emails, I need to create synthetic data. Hence, I want
    you to invent 3 coherent emails (and responses), similar to 
    the examples that have been given to you.

    # Output Format
    I want you to output these emails as an array of json objects
    formatted as such: 
    {{"question": "the user's request email", "response": "the customer support response"}}

    The final output should contain nothing more than the json output
    which must strictly adhere to the following schema: 
    {{"data": [
        {{"question": "1st question", "response": "response to 1st question"}}, 
        {{"question": "2nd question", "response": "response to 2nd question"}}, 
        {{"question": "3rd question", "response": "response to 3rd question"}}, 
        ]
    }}

    # Examples
    {examples}
    """
)

corpus = pd.read_parquet("./corpus_clean.parquet")

In [42]:
shard     = corpus[corpus["action"] == 'EXPLAIN_FAQ']
qa        = shard[["question", "response"]]
smpl      = qa.sample(3)

examples  = []
for i,row in smpl.iterrows():
    examples.append(QA(question=row["question"], response=row["response"]))

examples = json.dumps(examples, default=pydantic_encoder)

output = (template | gpt | JsonOutputParser()).invoke({"examples":examples})
output = ModelResponse.parse_obj(output).data

/tmp/ipykernel_3983/4251193819.py:12: PydanticDeprecatedSince20: The `parse_obj` method is deprecated; use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  output = ModelResponse.parse_obj(output).data


In [43]:
for x in output:
    print(x.question)
    print("$$" * 40)
    print(x.response)
    print("##"*40)

**Van:** jeroen.klaasen@nlmail.com <jeroen.klaasen@nlmail.com>
**Verzonden:** maandag 14 augustus 2024 09:12
**Aan:** digita <digita@arch.be>
**Onderwerp:** Vraag: geboorteaktes 1922


Beste,

Ik wil graag een originele geboortearchief voor mijn grootvader, Jan
Klaasen, geboren 12 maart 1922 in Brugge. Ik ben benieuwd of ik via de
website een digitaal exemplaar kan aanvragen of dat ik een fysieke
koppie moet opsturen. In welke taal moet het worden?

Met vriendelijke groet,
Jeroen Klaasen
$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$
Date: Wed, 16 Aug 2024 10:30:00 +0200
To: jeroen.klaasen@nlmail.com
From: digita <MAILER-DAEMON>
Subject: Re: Vraag: geboorteaktes 1922

Beste Jeroen,

Voor akten uit 1922 zijn uitsluitend digitale kopieën beschikbaar via de
Rijksarchief‑website. De originele akte blijft bij de gemeente Brugge en
kan niet via de centrale archiefinstelling worden uitgegeven. U kunt een
digitale reproduces aanvragen op de pagina van de Brugs